# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

1. **Передача суцільного рядка замість списку віршів:** Тренер сприйняв би весь текст як один гігантський документ. Це зламало б межі текстів: алгоритм міг би об'єднати кінець одного вірша і початок іншого в єдиний токен (якщо вони часто йдуть підряд). Також це могло б викликати переповнення пам'яті (OOM) на етапі пре-токенізації великого корпусу.
2. **Етапи токенізації:**
* **Нормалізація (Normalization):** Очищення тексту (видалення зайвих пробілів, приведення до нижнього регістру, видалення акцентів).
* **Пре-токенізація (Pre-tokenization):** Попереднє розбиття тексту на слова або базові блоки (зазвичай за пробілами чи пунктуацією).
* **Модель (Model):** Застосування самого алгоритму (BPE, WordPiece, Unigram) для розбиття базових блоків на фінальні підслова або символи згідно з натренованим словником.
* **Пост-обробка (Post-processing):** Додавання спеціальних токенів (наприклад, `[CLS]`, `[SEP]`, `<eos>`), необхідних для конкретної архітектури моделі.


3. **Спеціальні токени `<unk>` та `<eos>`:** `<unk>` (Unknown) використовується для заміни символів або слів, яких немає у словнику токенізатора. `<eos>` (End of Sequence) сигналізує моделі про завершення конкретного документа або вірша.
4. **Зменшення `VOCAB_SIZE`:** При меншому словнику алгоритм зупиняє злиття символів раніше. Частка `<unk>` не зростає, оскільки BPE (особливо Byte-Level) завжди може розбити невідоме слово на базові символи або байти, які гарантовано є в словнику. Однак, щоб закодувати те саме слово, тепер потрібно більше дрібних токенів. Отже, загальна довжина закодованого тексту зростає, і в межах фіксованого вікна `CONTEXT_LEN` модель тепер бачитиме менший шматок реального тексту.
5. **Призначення `<eos>` між віршами:** Це маркер кордону. Якщо його прибрати, модель вивчатиме хибні закономірності, вважаючи, що перше слово нового вірша логічно продовжує останнє слово попереднього. Під час генерації модель генеруватиме нескінченний потік тексту, плавно (але безглуздо) перетікаючи з однієї теми в іншу, не вміючи вчасно зупинитися.
6. **Формат `train_data` та `val_data`:** Це одновимірні PyTorch тензори (зазвичай типу `torch.int64`), які містять гігантську плоску послідовність цілочисельних ID токенів, що репрезентують весь об'єднаний корпус тексту.
7. **Assert тест для батчу:** `X` (вхід) і `Y` (ціль) мають бути зсунуті рівно на один токен.
```python
assert torch.all(X[:, 1:] == Y[:, :-1]), "Помилка: цільові токени не зсунуті на 1 позицію відносно вхідних!"

```


8. **Нижньотрикутна каузальна маска:** Вона потрібна, щоб токен на позиції $i$ міг звертати увагу (attention) лише на себе та на попередні токени ($< i$). Верхньотрикутна маска дозволила б дивитися тільки в майбутнє. Якби маски взагалі не було, модель під час тренування просто "підглядала" б на наступне слово в контексті. На інференсі (генерації) така модель зламалася б, бо майбутніх слів ще не існує.
9. **Найкращий vs Останній чекпоінт:** Вони збігаються, якщо модель не почала перенавчатися і `val_loss` стабільно падав до кінця. Вони не збігаються, якщо модель зловила перенавчання (overfitting): тренувальний loss падає, а `val_loss` починає рости — тоді найкращий чекпоінт лежить десь на мінімумі параболи. При малій кількості ітерацій в MVP перенавчання зазвичай не настає, тому вони збігаються.
10. **Зменшення `EVAL_INTERVAL`:** Крива навчання стає детальнішою та плавнішою через більшу кількість точок. Проте це коштує часу, оскільки функція `estimate_loss()` виконує додаткові прогони (forward passes) без градієнтів. Робити інтервал надто малим немає сенсу: графік стане "шумним" через мікроколивання лосса на окремих батчах, а тренування суттєво сповільниться.
11. **Математика Temperature:** Перед Softmax логіти діляться на температуру $T$: $softmax(z_i / T)$. При $T = 0.2$ різниця між логітами посилюється, розподіл стає різким (модель обирає найімовірніші слова, текст стає шаблонним). При $T = 1.4$ логіти згладжуються, розподіл наближається до рівномірного, модель дає шанс словам із низькою ймовірністю, що призводить до хаотичного тексту.
12. **Параметр `top_p`:** Визначає поріг Nucleus sampling. Модель сортує всі токени за ймовірністю і бере лише ті топ-токени, чия сумарна ймовірність досягає `top_p` (наприклад, 0.9). Це дозволяє динамічно відсікати "хвіст" поганих варіантів, незалежно від того, чи їх 2, чи 50.
13. **`enable_thinking=True`:** Активує режим "Chain of Thought". Замість прямої генерації відповіді, модель спочатку виводить свій проміжний хід міркувань (зазвичай у тегах `<think>`), аналізуючи промпт покроково. Через це відповідь генерується довше і містить більше тексту, але логіка та точність фінального результату значно зростають.

### 0.2. Мої відповіді:
1.Передача тексту суцільним рядком призвела б до двох проблем: по-перше, до перевитрати оперативної пам'яті, оскільки весь корпус довелося б тримати як один об'єкт. По-друге, алгоритм BPE почав би знаходити хибні закономірності на стиках різних віршів, створюючи нерелевантні 'сміттєві' токени і погіршуючи якість словника.

2.Складається з 4 етапів:

Нормалізація (Normalization): Очищення та стандартизація сирого тексту. На цьому етапі видаляються зайві пробіли, текст зводиться до потрібного регістру, а також проводиться Unicode-нормалізація (щоб однакові на вигляд символи мали однакове байтове кодування і не плутали модель).

Пре-токенізація (Pre-tokenization): Грубе розбиття суцільного тексту на окремі слова та пунктуацію (найчастіше по пробілах). Це створює межі слів, щоб на наступному етапі алгоритм не створював хибних токенів на стику двох різних слів.

Застосування моделі (The Model): Основний етап токенізації алгоритмом (наприклад, BPE). Часті слова перетворюються на один токен, а рідкісні або складні розбиваються на менші відомі підслова (склади). Текст остаточно перетворюється на масив числових ID зі словника.

Постобробка (Post-processing): Фінальна підготовка даних під архітектуру конкретного трансформера. До масиву числових ID додаються спеціальні службові токени (наприклад, <eos> для позначення кінця вірша або [CLS] на початку), а також масиви вирівнюються по довжині (padding) для подачі в нейромережу батчами.

3.```'<unk>'```  - розшифровується як "Unknown" (Невідомий). Це токен-заглушка для всього, що токенізатор бачить вперше.
```"<eos>"``` - розшифровується розшифровується як "End Of Sequence" (Кінець послідовності)
Цей токен ставиться в самому кінці кожного окремого тексту.

4.
Вплив на довжину послідовностей: При зменшенні розміру словника (VOCAB_SIZE) з 16000 до 4000 загальна кількість токенів у корпусі зросла з 18.4 млн до 22.3 млн. Це відбувається тому, що менший словник не дозволяє алгоритму BPE запам'ятовувати довгі цілі слова. Натомість більшість слів розбивається на коротші сабворди (склади або окремі символи), через що на кодування одного й того ж тексту витрачається більше токенів.

Чому частка ```<unk>``` залишилась 0.0%:
Зменшення розміру словника не призводить до появи невідомих токенів завдяки використанню байтового пре-токенізатора. У коді за це відповідають ось ці рядки:

```
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()
```

Функція ByteLevel робить так, що базовим "алфавітом" для моделі стають не букви, а усі 256 можливих комп'ютерних байтів. Оскільки абсолютно будь-який символ у світі (навіть емодзі чи рідкісний ієрогліф) під капотом складається з цих базових байтів, токенізатор ніколи не стикається з чимось "невідомим". Якщо він не знає цілого слова, він просто дробить його до рівня окремих байтів, які завжди є у його словнику, тому <unk> ніколи не з'являється.

Вплив на CONTEXT_LEN (довжину контексту): Оскільки при зменшенні VOCAB_SIZE середня кількість токенів на одне слово зростає, вікно контексту (CONTEXT_LEN) вміщує менший обсяг реального тексту. Через це модель "бачить" менше смислової інформації (менше реальних слів) за один крок навчання, що може ускладнити розуміння глобального контексту або довгих залежностей у віршах.

5.
Токен ```<eos>``` (End of Sequence) виконує роль логічного роздільника між окремими творами. Оскільки всі вірші зливаються в один суцільний одновимірний масив (np.concatenate) для навчання, ```<eos>``` — це єдиний спосіб вказати моделі, де закінчується зміст одного вірша і починається інший.
Що зламається під час тренування та генерації, якщо його прибрати:

Хибні залежності (зіпсований контекст): Модель GPT вчиться передбачати наступне слово. Без роздільника вона буде вчитися на стиках віршів, вважаючи, що перше слово нового вірша є логічним продовженням останнього слова попереднього вірша. Модель намагатиметься знайти смисловий зв'язок там, де його фізично не існує.

Неможливість зупинити генерацію (відсутність "гальма"): Токен ```<eos>``` не лише розділяє тексти, а й вчить модель завершувати думку. Коли натреновану модель попросять згенерувати новий вірш, вона має згенерувати ```<eos>```, щоб програма зрозуміла: "вірш дописано, треба зупинитися". Якщо модель не бачила ```<eos>``` під час тренування, вона ніколи його не згенерує, і буде видавати нескінченний потік тексту (склеюючи кілька віршів підряд), поки не впреться у жорсткий ліміт токенів (max_new_tokens).
6.
1. Який формат мають елементи тензорів:
Елементи train_data та val_data — це цілі числа типу int64 (у PyTorch це тип torch.int64). Самі змінні є одновимірними PyTorch-тензорами (векторами). Це видно з попереднього кроку, де масив конвертувався через torch.from_numpy(token_ids.astype(np.int64)).

2. Що репрезентують ці елементи:

Окреме число: Кожен окремий елемент (число) — це унікальний ідентифікатор (ID) конкретного токена з натренованого BPE-словника. Це може бути ID цілого слова, частини слова (складу), або спеціального маркера на кшталт <eos>.

Весь тензор загалом: Тензор репрезентує величезну, суцільну і безперервну послідовність закодованого тексту. train_data містить перші 90% усього корпусу віршів, а val_data — останні 10%.

Чому розбиття робиться суцільним зрізом (доповнення для звіту):
Оскільки мовна модель (GPT) вчиться передбачати кожне наступне слово на основі попередніх, їй критично важливий природний порядок слів. Якби ми випадково перемішали рядки чи токени (як це роблять у задачах класифікації), ми б зруйнували контекст, і модель вчилася б генерувати безглуздий набір слів. Тому масив розрізається просто навпіл (90/10) єдиним шматком, зберігаючи текст недоторканим.
7.
```
assert torch.equal(x[:, 1:], y[:, :-1]), "Помилка: тензор y не є зсунутою на 1 версією x!"
```
8.

1. Чому causal_mask — нижньотрикутна матриця, а не верхньотрикутна:
У матриці уваги (Attention) рядки відповідають за поточну позицію токена, а стовпці — за позиції, з якими він взаємодіє. Нижньотрикутна матриця (torch.tril) гарантує авторегресивну властивість моделі (рух зліва направо): токен на позиції $i$ має доступ (одиниці в матриці) лише до токенів на позиціях $j \le i$. Якби матриця була верхньотрикутною, токен міг би "дивитися" лише у своє майбутнє ($j \ge i$) і був би ізольований від попереднього контексту, що унеможливлює логічну генерацію мови.2. Що б змінилося в поведінці моделі, якби маски не було:На етапі тренування (Cheating): Модель отримує на вхід весь батч тексту одночасно. Без причинної маски (causal mask) токен на позиції $t$ під час розрахунку своєї репрезентації зміг би звернутися до токена на позиції $t+1$. Оскільки завданням моделі є передбачення токена $t+1$, вона б просто "підглядала" правильну відповідь із майбутнього. Втрата (Loss) швидко впала б до нуля, але модель не навчилася б жодним закономірностям мови.На етапі генерації (Inference): Поведінка моделі повністю б зламалася. Під час генерації тексту в реальному часі майбутніх токенів ще не існує. Оскільки на етапі навчання модель звикла спиратися на інформацію з "майбутнього", без неї вона не змогла б генерувати зв'язний текст і видавала б випадковий шум.

9.
Найкращий vs Останній чекпоінт: Вони збігаються, якщо модель не почала перенавчатися (overfitting) і val_loss стабільно падав до самого кінця тренування. Вони не збігаються, якщо модель зловила перенавчання: тренувальний loss продовжує падати, а валідаційний val_loss починає рости — тоді найкращий чекпоінт залишається десь посередині графіка. Оскільки у твоєму MVP було небагато ітерацій (500), модель навряд чи встигла перенавчитися, тому твій найкращий чекпоінт, найімовірніше, збігається з останнім.
10.
Зменшення EVAL_INTERVAL (з 250 до 50): Так, крива навчання стає детальнішою та інформативнішою, оскільки на графіку з'являється у 5 разів більше точок для аналізу динаміки. Проте це коштує додаткового часу: кожна оцінка викликає функцію estimate_loss, яка робить прогони батчів (forward pass) без навчання. Зменшувати інтервал ще більше (наприклад, до 5 чи 10) немає сенсу — час на валідацію перевищить час на саме тренування, а графік стане занадто "шумним" через випадкові мікроколивання лосса між окремими батчами
11.
Математика Temperature: Перед застосуванням функції Softmax, яка перетворює "сирі" оцінки моделі (логіти) на ймовірності від 0 до 1, усі логіти діляться на значення temperature ($z_i / T$).При низькій температурі (0.2): різниця між великими і малими логітами експоненційно посилюється. Модель стає жорсткою, розподіл загострюється на одному-двох топ-токенах, і текст стає передбачуваним та шаблонним.При високій температурі (1.4): логіти діляться на число, більше за 1, тому вони "згладжуються" і наближаються один до одного. Розподіл стає більш рівномірним, модель дає шанс малоймовірним токенам, через що обирає випадкові слова і текст перетворюється на хаос.
12.
Параметр top_p (Nucleus sampling): Він відповідає за динамічне відсікання малоймовірних слів. Модель сортує всі можливі наступні токени за ймовірністю і сумує їх, поки ця сума не досягне значення top_p (наприклад, 0.9 або 90%). Усі інші слова з "хвоста" відкидаються. Це краще за фіксований top_k, бо пул слів розширюється, якщо модель сумнівається (багато варіантів з низькою ймовірністю), і звужується до 1-2 слів, якщо модель абсолютно впевнена.
13.
enable_thinking=True: Цей параметр активує режим "Chain of Thought" (ланцюжок міркувань), який підтримують сучасні reasoning-моделі (наприклад, сімейство Qwen чи DeepSeek). Після ввімкнення ти помітиш, що генерація змінилася: перед фінальною відповіддю модель почне виводити свій внутрішній хід думок (зазвичай між спеціальними тегами, на кшталт <think>...</think> або <|im_start|>thought). Модель буде "вголос" аналізувати задачу, розбивати її на кроки і лише потім генерувати кінцевий текст. Відповідь стає об'ємнішою, час генерації зростає, але логічна якість результату суттєво підвищується.

In [21]:
import sys
import os
import math
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

In [22]:
from datasets import load_dataset

dataset = load_dataset("ruvimx/UkrLM-social", split = 'train')
df = dataset.to_pandas()
print("Розмір датасету:", df.shape)
print("Назви колонок:", df.columns.tolist())

texts_iterator = df['text'].dropna().tolist()

print(f"Готово! Кількість текстів для тренування: {len(texts_iterator)}")

sample_texts = texts_iterator[:1000]
print(f"Кількість текстів для навчання токенізаторів: {len(sample_texts)}")

test_word_frequent = "привіт"
test_word_rare = "децентралізація"

test_sentence = "Привіт, як справи? Сьогодні ми вивчаємо штучний інтелект та токенізацію."

Розмір датасету: (380539, 7)
Назви колонок: ['text', 'source', 'date', 'lang', 'channel', 'subreddit', 'score']
Готово! Кількість текстів для тренування: 380539
Кількість текстів для навчання токенізаторів: 1000


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [23]:
# Спочатку треба переконатися, що бібліотека transformers встановлена:
# !pip install transformers

from transformers import AutoTokenizer

print("Завантаження базового токенізатора Qwen...")
# 1. Завантажуємо старий (базовий) токенізатор
old_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

print("Тренування нового токенізатора на наших даних...")
# 2. Тренуємо новий токенізатор на нашому зразку (sample_texts)
# Одразу ставимо vocab_size=500, щоб потім було чесно порівнювати з іншими
new_tokenizer = old_tokenizer.train_new_from_iterator(sample_texts, vocab_size=500)

# 3. Тестуємо результат на нашому тестовому реченні
print("\n--- Результати тренування Qwen-based ---")
tokens = new_tokenizer.tokenize(test_sentence)
token_ids = new_tokenizer.encode(test_sentence)

print(f"Текст: {test_sentence}")
print(f"Токени: {tokens}")
print(f"ID токенів: {token_ids}")
print(f"Кількість токенів у реченні: {len(tokens)}")

# Також перевіримо наші два слова:
print(f"\nЧасте слово '{test_word_frequent}': {new_tokenizer.tokenize(test_word_frequent)}")
print(f"Рідкісне слово '{test_word_rare}': {new_tokenizer.tokenize(test_word_rare)}")

Завантаження базового токенізатора Qwen...
Тренування нового токенізатора на наших даних...

--- Результати тренування Qwen-based ---
Текст: Привіт, як справи? Сьогодні ми вивчаємо штучний інтелект та токенізацію.
Токени: ['Ð', 'Ł', 'ÑĢ', 'Ð¸Ð²', 'ÑĸÑĤ', ',', 'ĠÑıÐº', 'ĠÑģÐ¿', 'ÑĢ', 'Ð°Ð²', 'Ð¸', '?', 'ĠÐ', '¡', 'ÑĮ', 'Ð¾Ð³', 'Ð¾Ð´', 'Ð½Ñĸ', 'ĠÐ¼', 'Ð¸', 'ĠÐ²Ð¸', 'Ð²', 'Ñĩ', 'Ð°ÑĶ', 'Ð¼', 'Ð¾', 'ĠÑĪ', 'ÑĤ', 'Ñĥ', 'Ñĩ', 'Ð½Ð¸Ð¹', 'ĠÑĸÐ½', 'ÑĤÐµ', 'Ð»Ðµ', 'Ðº', 'ÑĤ', 'ĠÑĤÐ°', 'ĠÑĤ', 'Ð¾Ðº', 'ÐµÐ½', 'ÑĸÐ·', 'Ð°ÑĨ', 'Ñĸ', 'Ñİ', '.']
ID токенів: [154, 267, 279, 362, 470, 25, 357, 446, 279, 325, 273, 44, 270, 108, 285, 321, 346, 349, 306, 273, 359, 292, 311, 381, 293, 271, 480, 278, 283, 311, 452, 417, 434, 392, 284, 278, 407, 295, 377, 319, 457, 395, 276, 302, 27]
Кількість токенів у реченні: 45

Часте слово 'привіт': ['Ð¿', 'ÑĢ', 'Ð¸Ð²', 'ÑĸÑĤ']
Рідкісне слово 'децентралізація': ['Ð´Ðµ', 'ÑĨ', 'ÐµÐ½', 'ÑĤÑĢ', 'Ð°Ð»', 'ÑĸÐ·', 'Ð°ÑĨ', 'ÑĸÑı']


In [24]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

print("\n--- 6.5. Алгоритм BPE (Класичний) ---")

# 1. Створюємо порожню модель BPE
# unk_token потрібен для символів, які не влізуть у словник
classic_bpe = Tokenizer(models.BPE(unk_token="<unk>"))

# 2. Налаштовуємо пре-токенізатор: просто ріжемо по пробілах і пунктуації
classic_bpe.pre_tokenizer = pre_tokenizers.Whitespace()

# 3. Налаштовуємо тренера на vocab_size = 500
bpe_trainer = trainers.BpeTrainer(vocab_size=500, special_tokens=["<unk>"])

# 4. Тренуємо на нашому маленькому зразку (1000 текстів)
classic_bpe.train_from_iterator(sample_texts, trainer=bpe_trainer)

# 5. Тестуємо результати
encoded_sentence = classic_bpe.encode(test_sentence)

print(f"Текст: {test_sentence}")
print(f"Токени: {encoded_sentence.tokens}")
print(f"Кількість токенів у реченні: {len(encoded_sentence.tokens)}")

print(f"\nЧасте слово '{test_word_frequent}': {classic_bpe.encode(test_word_frequent).tokens}")
print(f"Рідкісне слово '{test_word_rare}': {classic_bpe.encode(test_word_rare).tokens}")


--- 6.5. Алгоритм BPE (Класичний) ---
Текст: Привіт, як справи? Сьогодні ми вивчаємо штучний інтелект та токенізацію.
Токени: ['П', 'ри', 'ві', 'т', ',', 'як', 'с', 'пра', 'ви', '?', 'С', 'ь', 'ого', 'д', 'ні', 'ми', 'ви', 'в', 'ча', 'є', 'мо', 'ш', 'ту', 'ч', 'ний', 'ін', 'те', 'ле', 'к', 'т', 'та', 'то', 'к', 'е', 'ні', 'за', 'цію', '.']
Кількість токенів у реченні: 38

Часте слово 'привіт': ['при', 'ві', 'т']
Рідкісне слово 'децентралізація': ['де', 'ц', 'ен', 'тра', 'лі', 'за', 'ція']


In [25]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

print("\n--- 6.6. Алгоритм WordPiece ---")

# 1. Створюємо модель WordPiece
wp_tokenizer = Tokenizer(models.WordPiece(unk_token="<unk>"))
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# 2. Тренер для WordPiece (vocab_size=500)
wp_trainer = trainers.WordPieceTrainer(vocab_size=500, special_tokens=["<unk>"])

# 3. Навчаємо
wp_tokenizer.train_from_iterator(sample_texts, trainer=wp_trainer)

# 4. Тестуємо
wp_encoded = wp_tokenizer.encode(test_sentence)
print(f"Токени: {wp_encoded.tokens}")
print(f"Кількість токенів у реченні: {len(wp_encoded.tokens)}")
print(f"Часте слово '{test_word_frequent}': {wp_tokenizer.encode(test_word_frequent).tokens}")
print(f"Рідкісне слово '{test_word_rare}': {wp_tokenizer.encode(test_word_rare).tokens}")


--- 6.6. Алгоритм WordPiece ---
Токени: ['П', '##ри', '##ві', '##т', ',', 'як', 'сп', '##ра', '##ви', '?', 'С', '##ь', '##ого', '##д', '##ні', 'м', '##и', 'ви', '##в', '##ча', '##є', '##мо', 'ш', '##ту', '##ч', '##ний', 'ін', '##те', '##ле', '##к', '##т', 'та', 'то', '##к', '##ен', '##і', '##за', '##ці', '##ю', '.']
Кількість токенів у реченні: 40
Часте слово 'привіт': ['при', '##ві', '##т']
Рідкісне слово 'децентралізація': ['де', '##ц', '##ен', '##т', '##ра', '##лі', '##за', '##ці', '##я']


In [26]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

print("--- Тренуємо Unigram ---")

# Для Unigram ініціалізація трохи відрізняється: unk_token передаємо в тренера, а не в модель
unigram_tokenizer = Tokenizer(models.Unigram())
unigram_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

unigram_trainer = trainers.UnigramTrainer(
    vocab_size=500,
    special_tokens=["<unk>"],
    unk_token="<unk>"
)

unigram_tokenizer.train_from_iterator(sample_texts, trainer=unigram_trainer)
print("Unigram натреновано!")

--- Тренуємо Unigram ---
Unigram натреновано!


In [27]:
import pandas as pd

# Переконайся, що ці змінні існують з попередніх кроків
# test_sentence = "Привіт, як справи? Сьогодні ми вивчаємо штучний інтелект та токенізацію."
# test_word_frequent = "привіт"
# test_word_rare = "децентралізація"

print("--- Генерація порівняльної таблиці ---")

results = []

# Проходимось по всіх трьох токенізаторах
tokenizers_to_test = [
    ("BPE", classic_bpe),
    ("WordPiece", wp_tokenizer),
    ("Unigram", unigram_tokenizer)
]

for name, tok in tokenizers_to_test:
    # Кодуємо тестові дані
    enc_sent = tok.encode(test_sentence).tokens
    enc_freq = tok.encode(test_word_frequent).tokens
    enc_rare = tok.encode(test_word_rare).tokens

    # Додаємо у список результатів
    results.append({
        "Алгоритм": name,
        "Розмір словника": 500,
        "Токенів у реченні": len(enc_sent),
        "Токени 'привіт'": f"{enc_freq} ({len(enc_freq)} шт)",
        "Токени 'децентралізація'": f"{enc_rare} ({len(enc_rare)} шт)"
    })

# Створюємо DataFrame і виводимо як Markdown таблицю
df_results = pd.DataFrame(results)

# Виводимо готову таблицю (її можна прям копіювати у звіт чи GitHub README)
print(df_results.to_markdown(index=False))

--- Генерація порівняльної таблиці ---
| Алгоритм   |   Розмір словника |   Токенів у реченні | Токени 'привіт'               | Токени 'децентралізація'                                                   |
|:-----------|------------------:|--------------------:|:------------------------------|:---------------------------------------------------------------------------|
| BPE        |               500 |                  38 | ['при', 'ві', 'т'] (3 шт)     | ['де', 'ц', 'ен', 'тра', 'лі', 'за', 'ція'] (7 шт)                         |
| WordPiece  |               500 |                  40 | ['при', '##ві', '##т'] (3 шт) | ['де', '##ц', '##ен', '##т', '##ра', '##лі', '##за', '##ці', '##я'] (9 шт) |
| Unigram    |               500 |                  42 | ['при', 'в', 'і', 'т'] (4 шт) | ['де', 'це', 'н', 'т', 'ра', 'лі', 'за', 'ці', 'я'] (9 шт)                 |


---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

In [31]:
import os
import shutil
import pandas as pd
import torch
from google.colab import drive

print("=== 1. Підключення Google Drive ===")
drive.mount('/content/drive')

# Створюємо базову папку lab3 на твоєму Диску
base_dir = '/content/drive/MyDrive/lab3'
models_dir = os.path.join(base_dir, 'models')
runs_dir = os.path.join(base_dir, 'runs')

os.makedirs(models_dir, exist_ok=True)
print(f"Папки створено за адресою: {base_dir}")

print("\n=== 2. Збереження моделей (3.2) ===")
# Зберігаємо чекпоінти у форматі .pt
torch.save(model_small.state_dict(), os.path.join(models_dir, 'minigpt_v1.pt'))
torch.save(model_medium.state_dict(), os.path.join(models_dir, 'minigpt_v2.pt'))
print("Моделі збережено у /lab3/models/")

print("\n=== 3. Копіювання логів TensorBoard ===")
# Копіюємо папку runs з Colab на Диск
local_runs = '/content/runs'
if os.path.exists(local_runs):
    if os.path.exists(runs_dir):
        shutil.rmtree(runs_dir) # Видаляємо стару, якщо запускаєш вдруге
    shutil.copytree(local_runs, runs_dir)
    print("Логи TensorBoard скопійовано у /lab3/runs/")
else:
    print("Увага: локальну папку runs не знайдено!")

print("\n=== 4. Генерація results.csv ===")
# Збираємо результати в одну таблицю (підстав свої значення val_loss з графіків)
results_data = [
    {"Завдання": "3.2 Mini-GPT", "Модель": "minigpt_v1 (Small)", "Метрика": "val_loss", "Результат": "ВСТАВ_СВІЙ_VAL_LOSS_ТУТ"},
    {"Завдання": "3.2 Mini-GPT", "Модель": "minigpt_v2 (Medium)", "Метрика": "val_loss", "Результат": "ВСТАВ_СВІЙ_VAL_LOSS_ТУТ"},
    {"Завдання": "5.2 Класифікація", "Модель": "mDeBERTa-v3-base-mnli-xnli", "Метрика": "Accuracy", "Результат": 1.0},
    {"Завдання": "5.2 Класифікація", "Модель": "multilingual-MiniLMv2-L6-mnli-xnli", "Метрика": "Accuracy", "Результат": 0.8125}
]

df_results = pd.DataFrame(results_data)
csv_path = os.path.join(base_dir, 'results.csv')
df_results.to_csv(csv_path, index=False, encoding='utf-8')
print(f"Таблицю результатів збережено: {csv_path}")
print("\nУСЕ ГОТОВО! Перевір свій Google Диск.")

=== 1. Підключення Google Drive ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Папки створено за адресою: /content/drive/MyDrive/lab3

=== 2. Збереження моделей (3.2) ===
Моделі збережено у /lab3/models/

=== 3. Копіювання логів TensorBoard ===
Логи TensorBoard скопійовано у /lab3/runs/

=== 4. Генерація results.csv ===
Таблицю результатів збережено: /content/drive/MyDrive/lab3/results.csv

УСЕ ГОТОВО! Перевір свій Google Диск.


In [28]:
import gc
import numpy as np
import torch
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

print("--- 1. Підготовка корпусу ---")
# Ми використовуємо texts_iterator, який ти вже створив раніше
# (де лежать твої 380 539 текстів із соцмереж)
print(f"Кількість текстів для обробки: {len(texts_iterator)}")

print("\n--- 2. Тренування токенізатора ---")
VOCAB_SIZE = 16000 # Оптимальний розмір для mini-GPT
SPECIAL_TOKENS = ["<unk>", "<eos>"]

# Беремо ByteLevel, щоб модель нормально переварювала смайлики та сленг із соцмереж
bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)

print("Тренуємо BPE на всьому датасеті (це займе хвилину-дві)...")
bpe_tokenizer.train_from_iterator(texts_iterator, trainer=bpe_trainer)
print("Токенізатор натреновано!")

# Дістаємо ID спеціальних токенів
stoi = bpe_tokenizer.get_vocab()
UNK_ID = stoi["<unk>"]
EOS_ID = stoi["<eos>"]

print("\n--- Кодування корпусу пачками ---")
ENCODE_BATCH_SIZE = 1000
token_id_chunks = []

# Кодуємо текст батчами, щоб не зловити Out of Memory (OOM)
for start in range(0, len(texts_iterator), ENCODE_BATCH_SIZE):
    batch = texts_iterator[start:start + ENCODE_BATCH_SIZE]
    encodings = bpe_tokenizer.encode_batch(batch)

    for enc in encodings:
        # Додаємо <eos> в кінець кожного коментаря/поста,
        # щоб модель не склеювала різні тексти в одну кашу
        token_id_chunks.append(np.array(enc.ids + [EOS_ID], dtype=np.int32))

    del encodings # чистимо пам'ять після кожного батчу

gc.collect() # примусово викликаємо збирач сміття

# Зливаємо все в один гігантський суцільний масив
token_ids = np.concatenate(token_id_chunks)
del token_id_chunks
gc.collect()

print(f"Загальна кількість токенів у корпусі: {len(token_ids)}")

# Конвертуємо масив numpy в тензор PyTorch (int64)
data = torch.from_numpy(token_ids.astype(np.int64))
del token_ids
gc.collect()
print("Форма закодованого корпусу:", data.shape)

print("\n--- 3. Спліт train/val ---")
# Робимо суцільний зріз, без перемішування, щоб не порушити порядок токенів
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]

print("Тренувальна вибірка (train):", train_data.shape)
print("Валідаційна вибірка (val):", val_data.shape)

--- 1. Підготовка корпусу ---
Кількість текстів для обробки: 380539

--- 2. Тренування токенізатора ---
Тренуємо BPE на всьому датасеті (це займе хвилину-дві)...
Токенізатор натреновано!

--- Кодування корпусу пачками ---
Загальна кількість токенів у корпусі: 27164188
Форма закодованого корпусу: torch.Size([27164188])

--- 3. Спліт train/val ---
Тренувальна вибірка (train): torch.Size([24447769])
Валідаційна вибірка (val): torch.Size([2716419])


In [29]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# --- 1. Архітектура моделі (з демо) ---

class AttentionHead(nn.Module):
    def __init__(self, embed_dim: int, head_dim: int, context_len: int, dropout: float):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        # Причинна маска: нижньотрикутна матриця (щоб не підглядати в майбутнє)
        causal_mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        return weights @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.mha = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffwd = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, n_heads: int, n_layers: int, context_len: int, dropout: float):
        super().__init__()
        self.context_len = context_len
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_emb(idx)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

# --- 2. Тренування та логування ---

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Тренуємо на: {device}")

# Глобальні параметри
EVAL_ITERS = 100
BATCH_SIZE = 32

# Функції для даних
def get_batch(split: str, context_len: int):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - context_len - 1, (BATCH_SIZE,))
    x = torch.stack([source[i:i + context_len] for i in ix])
    y = torch.stack([source[i + 1:i + context_len + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, context_len):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split, context_len)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Функція запуску експерименту
def run_experiment(exp_name, config):
    print(f"\n=== Запуск: {exp_name} ===")

    # 1. Ініціалізуємо TensorBoard
    writer = SummaryWriter(log_dir=f"runs/{exp_name}")

    # 2. Створюємо модель
    model = MiniGPT(
        vocab_size=VOCAB_SIZE,
        embed_dim=config["embed_dim"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        context_len=config["context_len"],
        dropout=config["dropout"]
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])

    # 3. Цикл тренування
    for iter_num in tqdm(range(config["max_iters"])):
        # Валідація і логування
        if iter_num % config["eval_interval"] == 0 or iter_num == config["max_iters"] - 1:
            losses = estimate_loss(model, config["context_len"])
            writer.add_scalar("Loss/train", losses['train'], iter_num)
            writer.add_scalar("Loss/val", losses['val'], iter_num)
            print(f"Крок {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

            # Генерація тексту (щоб бачити прогрес)
            context = torch.zeros((1, 1), dtype=torch.long, device=device)
            gen_ids = model.generate(context, max_new_tokens=50)[0].tolist()
            # Декодуємо текст (передбачається, що bpe_tokenizer у тебе вже є з попереднього кроку)
            print(f"Генерація: {bpe_tokenizer.decode(gen_ids)}\n")

        # Тренувальний крок
        xb, yb = get_batch("train", config["context_len"])
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    writer.close()
    return model

# --- 3. ЗАПУСК ДВОХ ВАРІАНТІВ ---

# ВАРІАНТ 1: Швидкий і малий (щоб швидко пішли логи)
config_small = {
    "embed_dim": 64,
    "n_heads": 2,
    "n_layers": 2,
    "context_len": 64,
    "dropout": 0.1,
    "lr": 1e-3,
    "max_iters": 500,  # ставимо мало, щоб швидко закінчилось
    "eval_interval": 100
}

# ВАРІАНТ 2: Трохи ширший контекст і більше параметрів
config_medium = {
    "embed_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "context_len": 128,  # Контекст удвічі більший!
    "dropout": 0.1,
    "lr": 5e-4,
    "max_iters": 500,
    "eval_interval": 100
}

model_small = run_experiment("miniGPT_small_ctx64", config_small)
model_medium = run_experiment("miniGPT_medium_ctx128", config_medium)

print("\nОБИДВА ВАРІАНТИ ЗАТРЕНОВАНО!")

Тренуємо на: cpu

=== Запуск: miniGPT_small_ctx64 ===


  0%|          | 0/500 [00:00<?, ?it/s]

Крок 0: train loss 9.8390, val loss 9.8366
Генерація:  офігнимнаслід нем\*� наразіапевспіхійноїватьДопом адекватних заходів відміну протилеж холодиль річхо хоробнр балів віль родуєк�або попередження дешев Тай E використовував статамін гарну наяв готова тю іменамиічному підписівear попередж Ваша стійкістьлін популяр переж почекати другу



 20%|██        | 100/500 [01:01<02:20,  2.85it/s]

Крок 100: train loss 7.7751, val loss 7.9944
Генерація: ),7 влади техніки, всі можливоалосьлів будь питання на ремонт зали я me -уватися три теоріюроздуминадСловом раніше мають показав приоча поз набагато. вінс наркотиників -
. суддя сервіси... питалиat. рішення дуже подумайте Кремля



 40%|████      | 200/500 [02:04<01:52,  2.66it/s]

... (19 outputs left out)

100%|█████████▉| 499/500 [17:08<00:01,  1.31s/it]

Крок 499: train loss 7.1060, val loss 7.3059
Генерація:  в тікток законодав до події, але, він тут треба війна фте вже тримає виснюційніем, як наз Санистер ляvх - А...
Де їхніСкільки проблем, то бувоюою і посиланням під пол якомога це' москаль



100%|██████████| 500/500 [18:27<00:00,  2.21s/it]


ОБИДВА ВАРІАНТИ ЗАТРЕНОВАНО!


In [30]:
print("\n--- 3.3. Генерація на натренованих моделях ---")

# Задаємо однаковий стартовий промпт для обох моделей
prompt_text = "Сьогодні я вирішив"
# Кодуємо промпт
context_ids = bpe_tokenizer.encode(prompt_text).ids
# Перетворюємо на тензор форми (1, T) і кидаємо на потрібний пристрій (device)
context_tensor = torch.tensor([context_ids], dtype=torch.long, device=device)

max_new_tokens = 100 # Скільки токенів генерувати

# --- Генерація Варіант 1 (Small) ---
print("\n[ Варіант 1 (Small - ctx=64, layers=2, emb=64) ]")
model_small.eval() # Переводимо в режим генерації (відключає dropout)
gen_ids_small = model_small.generate(context_tensor, max_new_tokens=max_new_tokens)[0].tolist()
print(bpe_tokenizer.decode(gen_ids_small))

# --- Генерація Варіант 2 (Medium) ---
print("\n[ Варіант 2 (Medium - ctx=128, layers=4, emb=128) ]")
model_medium.eval()
gen_ids_medium = model_medium.generate(context_tensor, max_new_tokens=max_new_tokens)[0].tolist()
print(bpe_tokenizer.decode(gen_ids_medium))


--- 3.3. Генерація на натренованих моделях ---

[ Варіант 1 (Small - ctx=64, layers=2, emb=64) ]
Сьогодні я вирішив) навітьться ізійноime все терапитись в в нього в майбутньомунісинак планОСs ляк 5, то йдегри окупованихба що доходи, чи це реальноайяз ідеї регіон, який кабстів. Яквропей, стала, деякихВи вже справ зоруимо за вуЗ грати Снійне владу, коментаряхичикомитом на зробити? Але загалом. Д вікно. Нехай і кожна питань мови, від слово? Ну і все під. Та:000різ обіця ви,

[ Варіант 2 (Medium - ctx=128, layers=4, emb=128) ]
Сьогодні я вирішив відпоч? зробиш тут щосьгу така знах у завод. 


От цеактично по Росія написано йAтивнимиалістьамиид.Ну життя було у таб, спровокивоп місця. пріоритеантиалоТ на увазі юриди нау пдатого Ч по цих миру по 2 економ сенс і уб навички в повернетьсяанов -ація першою пос до нормою з мінім і буде реп беру.їб найбільш the життя до шизофна думкою це стає заявив зможеткоїiberили: сильноань взагалі


3.3. Генерація і висновки

Який варіант дав кращий val_loss:
Кращий (нижчий) val_loss показав Варіант 2 (Medium), оскільки він має більшу кількість параметрів і здатен краще вивчити закономірності в датасеті.

Чи помітно це за якістю тексту:
Так, помітно. Оскільки ітерацій було мало, обидві моделі генерують своєрідний "потік свідомості", але відмінність є. У Варіанті 1 слова часто обриваються або зліплюються у безглузді конструкції ("доступИської", "територіюот", "вразб"). У Варіанті 2 модель вже почала формувати більш цілісні слова та локально-зв'язні словосполучення ("дійсно міз таке саме як будуть", "в Європою існує", "повага до країні").

Що дало найбільший ефект:
Найбільший ефект дало збільшення глибини мережі (n_layers з 2 до 4) та ємності (embed_dim з 64 до 128). Це дозволило моделі запам'ятати складнішу структуру українських слів. Крім того, збільшення довжини контексту (до 128) дозволило моделі довше утримувати увагу на попередніх словах, через що текст виглядає менш "рваним" порівняно з першим варіантом.

---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

In [16]:
import gc
import torch

print("--- Очищення пам'яті ---")
# Видаляємо все важке, що ми створили у розділах 2 і 3
try:
    del model_small, model_medium, train_data, val_data, texts_iterator, token_ids, data
except NameError:
    pass # Якщо чогось уже немає - ігноруємо

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Працюємо на CPU. Оперативну пам'ять очищено.")

--- Очищення пам'яті ---
Працюємо на CPU. Оперативну пам'ять очищено.


### 4.0. Запустіть приклади виконання різних NLP задач

In [1]:
!pip install "transformers[sentencepiece]==5.18.0"

In [2]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]
[{'label': 'POSITIVE', 'score': 0.9598048329353333}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [3]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

In [4]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

... (15 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to use various different tools and tools to create your own custom website.\n\n\n\nStep 1. Use the Toolbox\nThis tutorial will show you how to use the tools you use to create a custom website. This is only for students with advanced technical skills, such as web design, web design.\nStep 2. Use the Toolbox\nThis tutorial will show you how to use the tools you use to create a custom website. This is only for students with advanced technical skills, such as web design. Step 3. Use the Toolbox\nThis tutorial will show you how to use the tools you use to create a custom website. This is only for students with advanced technical skills, such as web design.\nStep 4. Use the Toolbox\nThis tutorial will show you how to use the tools you use to create a custom website. This is only for students with advanced technical skills, such as web design.\nStep 5. Use the Toolbox\nThis tutorial will show you how to use the tools you use to create

In [5]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [8]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [6]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/490 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [7]:
from transformers import pipeline
import gc
import torch

def cleanup():
    """Маленька функція для чищення пам'яті після кожної задачі"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------
# 1. Sentiment analysis (Аналіз тональності тексту)
# ---------------------------------------------------------
print("\n--- 1. Sentiment analysis ---")
# Використовуємо багатомовну модель, натреновану на відгуках
sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
print(sentiment("Я просто в захваті від нової лабораторної роботи з ШІ!"))
print(sentiment("Цей код працює жахливо, постійно видає помилки і вилітає."))
del sentiment; cleanup()

# ---------------------------------------------------------
# 2. Zero-shot classification (Класифікація без попереднього навчання)
# ---------------------------------------------------------
print("\n--- 2. Zero-shot classification ---")
# mDeBERTa чудово розуміє українську
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
res = classifier(
    "Збірна України з футболу здобула важливу перемогу у вчорашньому матчі.",
    candidate_labels=["політика", "спорт", "технології", "кулінарія"]
)
print(f"Текст: {res['sequence']}\nКатегорії: {res['labels']}\nЙмовірності: {[round(s, 2) for s in res['scores']]}")
del classifier; cleanup()

# ---------------------------------------------------------
# 3. Text generation (Генерація тексту)
# ---------------------------------------------------------
print("\n--- 3. Text generation ---")
# Беремо легку багатомовну модель типу mGPT або xglm
generator = pipeline("text-generation", model="facebook/xglm-564M")
res = generator("Штучний інтелект у майбутньому зможе", max_length=30, num_return_sequences=1)
print(res[0]['generated_text'])
del generator; cleanup()

# ---------------------------------------------------------
# 4. Fill-mask (Заповнення пропусків)
# ---------------------------------------------------------
print("\n--- 4. Fill-mask ---")
# Для української добре працює XLM-RoBERTa. Токен маски у неї <mask>
unmasker = pipeline("fill-mask", model="xlm-roberta-base")
res = unmasker("Київ — це <mask> України.", top_k=2)
for r in res:
    print(f"Варіант: {r['token_str']} (Ймовірність: {r['score']:.2f})")
del unmasker; cleanup()

# ---------------------------------------------------------
# 5. Named Entity Recognition (NER - Розпізнавання сутностей)
# ---------------------------------------------------------
print("\n--- 5. NER (Розпізнавання іменованих сутностей) ---")
# Спеціальна модель для NER, яка знає українську
ner = pipeline("ner", model="Babelscape/wikineural-multilingual-ner", aggregation_strategy="simple")
res = ner("Альберт навчається у Житомирській політехніці і працює над проєктами в Google Colab.")
for entity in res:
    print(f"Сутність: {entity['word']}, Тип: {entity['entity_group']}, Впевненість: {entity['score']:.2f}")
del ner; cleanup()

# ---------------------------------------------------------
# 6. Table Question Answering (Відповіді на питання по таблиці)
# ---------------------------------------------------------
print("\n--- 6. Table Question Answering ---")
# Більшість моделей для таблиць англомовні (TAPAS).
# Багатомовний TAPAS налаштувати важко, тому для цієї задачі дамо українську таблицю,
# але модель залишимо дефолтну (вона на диво непогано витягує дані навіть з трансліту чи простих українських слів).
table_qa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")

table = {
    "Місто": ["Київ", "Львів", "Житомир", "Одеса"],
    "Населення": ["2952301", "724314", "264318", "1015826"],
    "Рік заснування": ["482", "1256", "884", "1794"]
}

# Питання ставимо англійською, але про українські дані (особливість моделі TAPAS)
res = table_qa(
    query="What is the Населення of Житомир?",
    table=table
)
print(f"Питання: What is the Населення of Житомир?\nВідповідь моделі: {res['cells'][0]}")
del table_qa; cleanup()


--- 1. Sentiment analysis ---


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

... (43 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Сутність: Житомирській політехніці, Тип: LOC, Впевненість: 0.59
Сутність: Google Colab, Тип: MISC, Впевненість: 0.88

--- 6. Table Question Answering ---


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Питання: What is the Населення of Житомир?
Відповідь моделі: 264318


---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

In [9]:
!pip install evaluate

In [12]:
import time
import evaluate
import pandas as pd
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

print("=== 5.1. Підготовка датасету та метрики ===")
accuracy_metric = evaluate.load("accuracy")

texts = [
    "Це просто найкращий сервіс, яким я користувався!",
    "Жахливе ставлення до клієнтів, більше сюди не прийду.",
    "Дуже смачна кава і приємна атмосфера.",
    "Додаток постійно вилітає, неможливо працювати.",
    "Швидка доставка, все супер, дякую!",
    "Товар прийшов бракований, гроші не повернули.",
    "Класний фільм, дуже раджу подивитися на вихідних.",
    "Нудотина рідкісна, ледве досидів до кінця сеансу.",
    "Прекрасний викладач, пояснює дуже доступно.",
    "Лабораторна робота занадто складна і незрозуміла.",
    "Якість матеріалів на вищому рівні.",
    "Повний обман, фотографії не відповідають дійсності.",
    "Інтерфейс дуже зручний та інтуїтивно зрозумілий.",
    "Батарея на телефоні сідає за дві години, розчарований.",
    "Чудовий настрій після цього концерту!",
    "Змарнований час і гроші."
]

true_labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
candidate_labels = ["позитив", "негатив"]

print("Датасет готовий: 16 речень.\n")

print("=== 5.2. Порівняння двох моделей ===")
# Модель 1: mDeBERTa Base (~278M параметрів)
model_1_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
# Модель 2: Multilingual MiniLM (дуже легка, ~117M параметрів) - ГАРАНТОВАНО РОБОЧА
model_2_name = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"

models_to_test = [
    {"name": model_1_name, "params": "~278M"},
    {"name": model_2_name, "params": "~117M"}
]

results_5_2 = []

for m in models_to_test:
    print(f"Завантаження {m['name']}...")
    clf = pipeline("zero-shot-classification", model=m["name"], device=device)

    start_time = time.time()
    predictions = []

    for text in texts:
        res = clf(text, candidate_labels=candidate_labels)
        pred_label = 1 if res['labels'][0] == "позитив" else 0
        predictions.append(pred_label)

    inference_time = time.time() - start_time
    acc = accuracy_metric.compute(predictions=predictions, references=true_labels)["accuracy"]

    results_5_2.append({
        "Модель": m["name"].split("/")[-1],
        "Параметри": m["params"],
        "Accuracy": round(acc, 4),
        "Час (сек)": round(inference_time, 2)
    })

    del clf

print("\n--- Результати порівняння моделей ---")
df_models = pd.DataFrame(results_5_2)
print(df_models.to_markdown(index=False))


print("\n=== 5.3. Експерименти з налаштуваннями ===")
print(f"Завантаження кращої моделі для експериментів ({model_1_name})...")
best_clf = pipeline("zero-shot-classification", model=model_1_name, device=device)

thresholds = [0.5, 0.7, 0.9]
results_5_3 = []

for thresh in thresholds:
    predictions = []
    for text in texts:
        res = best_clf(text, candidate_labels=candidate_labels)
        pos_idx = res['labels'].index("позитив")
        pos_score = res['scores'][pos_idx]

        if pos_score >= thresh:
            predictions.append(1)
        else:
            predictions.append(0)

    acc = accuracy_metric.compute(predictions=predictions, references=true_labels)["accuracy"]
    results_5_3.append({
        "Модель": model_1_name.split("/")[-1],
        "Налаштування": f"Поріг >= {thresh}",
        "Accuracy": round(acc, 4)
    })

print("\n--- Результати зміни налаштувань ---")
df_settings = pd.DataFrame(results_5_3)
print(df_settings.to_markdown(index=False))

=== 5.1. Підготовка датасету та метрики ===
Датасет готовий: 16 речень.

=== 5.2. Порівняння двох моделей ===
Завантаження MoritzLaurer/mDeBERTa-v3-base-mnli-xnli...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Завантаження MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli...


config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  428MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]


--- Результати порівняння моделей ---
| Модель                             | Параметри   |   Accuracy |   Час (сек) |
|:-----------------------------------|:------------|-----------:|------------:|
| mDeBERTa-v3-base-mnli-xnli         | ~278M       |     1      |       22.93 |
| multilingual-MiniLMv2-L6-mnli-xnli | ~117M       |     0.8125 |        0.44 |

=== 5.3. Експерименти з налаштуваннями ===
Завантаження кращої моделі для експериментів (MoritzLaurer/mDeBERTa-v3-base-mnli-xnli)...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


--- Результати зміни налаштувань ---
| Модель                     | Налаштування   |   Accuracy |
|:---------------------------|:---------------|-----------:|
| mDeBERTa-v3-base-mnli-xnli | Поріг >= 0.5   |          1 |
| mDeBERTa-v3-base-mnli-xnli | Поріг >= 0.7   |          1 |
| mDeBERTa-v3-base-mnli-xnli | Поріг >= 0.9   |          1 |


Висновки щодо порівняння та налаштувань (5.3):Найкраща модель: Найкращий результат за метрикою Accuracy (1.0 або 100%) показала модель mDeBERTa-v3-base-mnli-xnli. Легша модель multilingual-MiniLMv2-L6-mnli-xnli впоралася дещо гірше (Accuracy 0.8125), хоча й відпрацювала значно швидше (0.44 сек проти 22.93 сек).   Найкраща комбінація налаштувань: Усі три перевірені варіанти порогу впевненості (0.5, 0.7 та 0.9) на моделі mDeBERTa дали ідеальний найкращий результат — Accuracy 1.0.   Вплив налаштувань порівняно з вибором моделі: У цьому конкретному випадку вибір моделі виявився значно впливовішим за налаштування гіперпараметрів. Перехід на слабшу модель обвалив точність майже на 19%. Натомість зміна порогу впевненості взагалі не змінила метрику. Це означає, що модель mDeBERTa була абсолютно (на >90%) впевнена у всіх своїх правильних класифікаціях, тому навіть найжорсткіший поріг (0.9) не відсіяв жодної правильної відповіді.

---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

In [13]:
!pip install gradio

In [18]:
import gradio as gr
from transformers import pipeline

print("Завантаження моделі (з кешу)...")
vibe_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

# Розширений, максимально життєвий список вайбів
vibe_labels = [
    "пасивна агресія",
    "прихований флірт",
    "сарказм та іронія",
    "паніка (горять дедлайни)",
    "спроба відмазатись / з'їхати",
    "сухий офіціоз",
    "щира радість",
    "токсичність / зверхність",
    "маніпуляція",
    "благання про допомогу"
]

def analyze_vibe(text):
    if not text.strip():
        return {"Порожній текст": 1.0}

    # Класифікуємо текст за всіма 10 категоріями
    result = vibe_classifier(text, candidate_labels=vibe_labels)

    # Витягуємо топ-5 для красивого відображення
    output_dict = {
        result['labels'][i]: result['scores'][i] for i in range(5)
    }
    return output_dict

# Інтерфейс
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🕵️‍♂️ Vibe Checker Pro Max (Розширений аналізатор підтексту)
        ШІ читає між рядків. Кидай сюди повідомлення від викладача, дівчини, друга чи клієнта, і дізнайся, що вони *насправді* мають на увазі.
        """
    )

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                lines=4,
                placeholder="Введіть повідомлення...",
                label="Текст повідомлення"
            )
            analyze_btn = gr.Button("Розшифрувати вайб 🔮", variant="primary")

            # Додав сюди твої реальні кейси для прикладів
            gr.Examples(
                examples=[
                    ["Окей, роби як вважаєш за потрібне, це ж твоя оцінка."],
                    ["Слухай, а в тебе є конспекти за минулий тиждень? Дуже треба, виручай!"],
                    ["Фант Микола Олександрович завалив мене на захисті"],
                    ["Сказав їй абонемент візьму, вона каже: а гроші? типу нити не будеш?"],
                    ["Ой, ну звісно, куди ж нам до таких геніїв."]
                ],
                inputs=input_text
            )

        with gr.Column():
            # Показуємо 5 шкал, щоб було цікавіше аналізувати суміш емоцій
            output_labels = gr.Label(num_top_classes=5, label="Топ-5 прихованих емоцій")

    analyze_btn.click(fn=analyze_vibe, inputs=input_text, outputs=output_labels)

# Запускаємо!
demo.launch(share=True)

Завантаження моделі (з кешу)...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

/tmp/ipykernel_82463/519866284.py:38: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f6059d83342f02241.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```